In [1]:
from gitsource import GithubRepositoryDataReader

# Fetch .md files from llm-zoomcamp lessons folder
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
### Q1

len(files), len(documents)

(72, 72)

In [4]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [5]:
### Q2

from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    # boost_dict={"question": 2.0, "section": 0.5},
    # filter_dict={"course": "llm-zoomcamp"},
    num_results=3
)

print(search_results[0]["filename"])

search_results

01-agentic-rag/lessons/14-agentic-loop.md


[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [6]:
from dotenv import load_dotenv
load_dotenv()

from rag_helper import RAGParsedDocs
from openai import OpenAI

assistant = RAGParsedDocs(index=index, llm_client=OpenAI())
response = assistant.rag("How does the agentic loop keep calling the model until it stops?")


In [7]:
## Q3
response.usage.input_tokens

7136

In [8]:
## Q4

from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [9]:
index_chunks = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index_chunks.fit(chunks)

assistant_chunks = RAGParsedDocs(index=index_chunks, llm_client=OpenAI())
response_chunks = assistant_chunks.rag("How does the agentic loop keep calling the model until it stops?")

In [10]:
## Q5

response_chunks.usage.input_tokens

2319

In [11]:
7136 / 2319

3.0771884432945233

In [ ]:
def search(query: str, num_results: int=5) -> list[dict[str, str]]:
    """Search the course lessons documentation for entries matching the given query."""

    return index_chunks.search(
        query,
        num_results=num_results,
    )

search("How does the agentic loop keep calling the model until it stops?")

[{'start': 4000,
  'content': 'while` loop. The loop keeps calling the model until\nit returns a response without any function calls. We also keep an\niteration counter so we can see how many round-trips happened.\n\n```python\nit = 1\n\nwhile True:\n    print(f"iteration #{it}...")\n    has_function_calls = False\n\n    response = openai_client.responses.create(\n        model="gpt-5.4-mini",\n        input=messages,\n        tools=[search_tool],\n    )\n\n    messages.extend(response.output)\n\n    for item in response.output:\n        if item.type == "function_call":\n            print("function_call:", item.name, item.arguments)\n            call_output = make_call(item)\n            messages.append(call_output)\n            has_function_calls = True\n\n        elif item.type == "message":\n            print("ASSISTANT:")\n            print(item.content[0].text)\n\n    it = it + 1\n    if has_function_calls == False:\n        break\n```\n\nThis is the core agent loop. The model rea

In [20]:
## Q6

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

agent_tools = Tools()
agent_tools.add_tool(search)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


In [23]:
runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.", role='developer', phase=None, type=None), EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"agentic loop RAG difference", "num_results": 5}', call_id='call_Z596BK8NXmC2jOarFNIO2TBg', name='search', type='function_call', id='fc_0102be7fba032883006a3a60f02a3081a1a02ac6fe61cc7429', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"agentic loop", "num_results": 5}', call_id='call_oAkTuZVx3IqWZbL23nE4WwCq', name='search', type='function_call', id='fc_0102be7fba032883006a3a60f02a4481a18823cce9772a2bcb', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"plain RAG", "num_results": 5}', call_id='

In [19]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course lessons documentation for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'},
    'num_results': {'type': 'number', 'description': 'num_results parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]